# Drift-aware few-shot spam detection - TREC 2007 emails
**Part A runs on CPU** (Runtime -> Change runtime type -> CPU). **Part B needs a T4 GPU.** Changing the runtime type restarts the session: just re-run the *Setup* cell of the new part; all results stay on Google Drive.

## Setup (run first in EVERY part)
Upload the newest `drift-email_*.zip` and `trec07p.tgz` to `My Drive` first.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip -q install transformers accelerate bitsandbytes scikit-learn scipy matplotlib
import glob, os
ZIP = max(glob.glob('/content/drive/MyDrive/drift-email*.zip'), key=os.path.getmtime); print('using', ZIP)
!unzip -q -o "$ZIP" -d /content
%cd /content/drift-email
WORK = '/content/drive/MyDrive/trec_work'
os.makedirs(WORK, exist_ok=True)
PARSED = WORK + '/trec07p_parsed.csv.gz'

In [ ]:
import os, glob, tarfile, zipfile

def _extract_all(src, dest):
    if zipfile.is_zipfile(src):
        with zipfile.ZipFile(src) as z: z.extractall(dest)
    elif tarfile.is_tarfile(src):
        with tarfile.open(src) as t:
            try: t.extractall(dest, filter="data")
            except TypeError: t.extractall(dest)
    else:
        raise RuntimeError("file not zip/tar: first bytes %r" % open(src, "rb").read(60))

def find_corpus(search_root="/content/drive/MyDrive", dest="/content/trec07p"):
    hits = lambda: glob.glob(dest + "/**/full/index", recursive=True)
    if not hits():
        cands = [c for c in glob.glob(search_root + "/**/*trec07p*", recursive=True) if os.path.isfile(c)]
        print("candidate files:", [(c, round(os.path.getsize(c) / 1e6, 1)) for c in cands])
        assert cands, "no file with 'trec07p' in its name under " + search_root
        os.makedirs(dest, exist_ok=True); _extract_all(cands[0], dest)
        for _ in range(3):                                    # nested archives, e.g. a .tgz inside a .zip
            if hits(): break
            for f in glob.glob(dest + "/**/*", recursive=True):
                if os.path.isfile(f) and f.lower().endswith((".tgz", ".tar", ".gz", ".zip")):
                    _extract_all(f, os.path.dirname(f)); os.rename(f, f + ".done")
    h = hits(); assert h, "extracted but full/index not found; contents: " + str(os.listdir(dest))
    return os.path.dirname(os.path.dirname(h[0]))

ROOT = find_corpus()
print("corpus ok:", ROOT, "| emails:", len(os.listdir(ROOT + "/data")))

# PART A (CPU)
### A1. Parse the corpus (about 10-20 minutes; saves to Drive). Skip if `trec07p_parsed.csv.gz` already exists.

In [ ]:
!python prep_trec.py --root $ROOT --out $PARSED

### A2. Label-free drift monitor: which weeks would DTR refresh? (about 2 minutes)
Copy the printed lines and send them to me BEFORE the GPU runs - I will choose the multipliers from them.

In [ ]:
!python monitor_schedule.py --parsed $PARSED --out $WORK/monitor

### A3. Classical baseline with the same label budgets (a few minutes)

In [ ]:
!python run_classical.py --parsed $PARSED --out $WORK/classical

# PART B (T4 GPU)
Runtime -> Change runtime type -> T4 GPU, then run the **Setup** cells again.
### B1. Speed pilot (zero-shot + static only, small sample). Send me the printed 'ms/msg'.

In [ ]:
!python run_email_drift.py --parsed $PARSED --out $WORK/pilot --only zs static --seeds 0 --eval_per_class 10 --no_4bit --batch_size 8

### B2. Full LLM experiment (resumable: if Colab disconnects, run the same cell again)
`--dtr_mult` values will be given to me from A2 - use the values I send.

In [ ]:
!python run_email_drift.py --parsed $PARSED --out $WORK/llm --no_4bit --batch_size 8 --dtr_mult 2 4 8

### B3. Base-paper reproduction (Spam-T5): download the datasets, then fine-tune Flan-T5-base

In [ ]:
import requests
rec = requests.get('https://zenodo.org/api/records/8339691', timeout=60).json()
files = {f['key']: f['links']['self'] for f in rec['files']}
print(sorted(files))

In [ ]:
import subprocess
for key in files:
    if any(s in key.lower() for s in ['ling', 'assas', 'enron']):
        subprocess.run(['wget', '-q', '-O', f'/content/{key}', files[key]]); print('downloaded', key)
!ls -lh /content/*.csv

In [ ]:
# Edit the three file names below to match the printed list (Ling / SpamAssassin / Enron)
!python reproduce_spamt5.py --csvs ling=/content/Ling.csv --ks 8 32 128 full --out $WORK/spamt5

# PART C (CPU or GPU)
### C1. Tables, tests and figures (send me the printed tables and the files in `analysis/`)

In [ ]:
!python analyze.py --dirs $WORK/llm $WORK/classical --out $WORK/analysis